In [1]:
import math
import torch


torch.manual_seed(123)

sample_n = 100
features = 5
split = math.floor(sample_n * 0.8)

x_0 = torch.randn(sample_n, features) + torch.tensor([0.324, 1.41, -2.91, 1.43, -2.45])
y_0 = torch.zeros(sample_n, dtype=torch.long)

x_1 = torch.randn(sample_n, features) + torch.tensor([-0.234, -2.34, 1.56, 1.45, -1.23])
y_1 = torch.ones(sample_n, dtype=torch.long)

x_train = torch.cat((x_0[:split, :], x_1[:split, :]), dim=0)
y_train = torch.cat((y_0[:split], y_1[:split]), dim=0)

x_test = torch.cat((x_0[split:,:], x_1[split:, :]), dim=0)
y_test = torch.cat((y_0[split:], y_1[split:]), dim=0)

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)


torch.Size([160, 5])
torch.Size([160])
torch.Size([40, 5])
torch.Size([40])


In [3]:
from torch.utils.data import Dataset

class SimpleDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __getitem__(self, index):
        return self.x[index], self.y[index]
    
    def __len__(self):
        return self.x.shape[0]


train_ds = SimpleDataset(x_train, y_train)
test_ds = SimpleDataset(x_test, y_test)

In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset = train_ds,
    batch_size = 4,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size = 4
)

In [5]:

import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_in, d_in*4),
            nn.ReLU(),
            nn.Linear(d_in*4, 2)
        )

    def forward(self, x):
        return self.layers(x)


In [6]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

model = SimpleMLP(features, 2)
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)


epochs = 100
for epoch in range(epochs):
    model.train()
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = F.cross_entropy(logits, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch+1:03d}/{epochs:03d} | Batch: {batch_idx+1:03d}/{len(train_loader):03d} | Loss: {loss:.2f}")

device: cpu
Epoch: 001/100 | Batch: 001/040 | Loss: 0.59
Epoch: 001/100 | Batch: 002/040 | Loss: 0.58
Epoch: 001/100 | Batch: 003/040 | Loss: 0.60
Epoch: 001/100 | Batch: 004/040 | Loss: 0.58
Epoch: 001/100 | Batch: 005/040 | Loss: 0.61
Epoch: 001/100 | Batch: 006/040 | Loss: 0.65
Epoch: 001/100 | Batch: 007/040 | Loss: 0.53
Epoch: 001/100 | Batch: 008/040 | Loss: 0.63
Epoch: 001/100 | Batch: 009/040 | Loss: 0.45
Epoch: 001/100 | Batch: 010/040 | Loss: 0.55
Epoch: 001/100 | Batch: 011/040 | Loss: 0.59
Epoch: 001/100 | Batch: 012/040 | Loss: 0.42
Epoch: 001/100 | Batch: 013/040 | Loss: 0.59
Epoch: 001/100 | Batch: 014/040 | Loss: 0.54
Epoch: 001/100 | Batch: 015/040 | Loss: 0.67
Epoch: 001/100 | Batch: 016/040 | Loss: 0.45
Epoch: 001/100 | Batch: 017/040 | Loss: 0.50
Epoch: 001/100 | Batch: 018/040 | Loss: 0.35
Epoch: 001/100 | Batch: 019/040 | Loss: 0.33
Epoch: 001/100 | Batch: 020/040 | Loss: 0.45
Epoch: 001/100 | Batch: 021/040 | Loss: 0.36
Epoch: 001/100 | Batch: 022/040 | Loss: 0.4

In [7]:
def compute_accuracy(model, dataloader):
    with torch.no_grad():
        model.eval()

        correct = 0.0
        num_samples = 0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            predictions = torch.argmax(logits, dim=1)
            compare = predictions == y
            correct += torch.sum(compare)
            num_samples += len(compare)

    return (correct / num_samples).item()





In [8]:
compute_accuracy(model, train_loader)

1.0

In [9]:
compute_accuracy(model, test_loader)

1.0

In [2]:
import torch
torch.cuda.is_available()

/home/cipherman/Studies/vlm-study/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


False

In [6]:
import torch

lr = 0.1
x = torch.tensor(5.0, requires_grad=True)
for _ in range(100):
    f = x ** 2
    print(f)
    f.backward()
    with torch.no_grad():
        x -= lr* x.grad
    x.grad.zero_()


tensor(25., grad_fn=<PowBackward0>)
tensor(16., grad_fn=<PowBackward0>)
tensor(10.2400, grad_fn=<PowBackward0>)
tensor(6.5536, grad_fn=<PowBackward0>)
tensor(4.1943, grad_fn=<PowBackward0>)
tensor(2.6844, grad_fn=<PowBackward0>)
tensor(1.7180, grad_fn=<PowBackward0>)
tensor(1.0995, grad_fn=<PowBackward0>)
tensor(0.7037, grad_fn=<PowBackward0>)
tensor(0.4504, grad_fn=<PowBackward0>)
tensor(0.2882, grad_fn=<PowBackward0>)
tensor(0.1845, grad_fn=<PowBackward0>)
tensor(0.1181, grad_fn=<PowBackward0>)
tensor(0.0756, grad_fn=<PowBackward0>)
tensor(0.0484, grad_fn=<PowBackward0>)
tensor(0.0309, grad_fn=<PowBackward0>)
tensor(0.0198, grad_fn=<PowBackward0>)
tensor(0.0127, grad_fn=<PowBackward0>)
tensor(0.0081, grad_fn=<PowBackward0>)
tensor(0.0052, grad_fn=<PowBackward0>)
tensor(0.0033, grad_fn=<PowBackward0>)
tensor(0.0021, grad_fn=<PowBackward0>)
tensor(0.0014, grad_fn=<PowBackward0>)
tensor(0.0009, grad_fn=<PowBackward0>)
tensor(0.0006, grad_fn=<PowBackward0>)
tensor(0.0004, grad_fn=<PowBac

In [7]:
x = 1
print(id(x))
x = x+1
print(id(x))

135827624957032
135827624957064


In [10]:
def add_one(x=0):
    return x + 1

def mul_two(x=0):
    return 2 * x

fun_list=[
    add_one,
    mul_two
]

In [11]:
fun_list

[<function __main__.add_one(x=0)>, <function __main__.mul_two(x=0)>]

In [ ]:
import torch
import math

n_samples = 100
features = 5
split = math.floor(n_samples * 0.9)


x_0 = torch.randn(n_samples, features) + torch.tensor([0.43, 1.567, -2.34, -2.45, 1.22])
y_0 = torch.zeros(n_samples, dtype=torch.long)

x_1 = torch.randn(n_samples, features) + torch.tensor([1.43, -2.45, -0.58, -1.23, 0.37])
y_1 = torch.ones(n_samples, dtype=torch.long)

x_train = torch.cat((x_0[:split, :], x_1[:split]), dim=0)
y_train = torch.cat((y_0[:split], y_1[:split]), dim=0)

x_test = torch.cat((x_0[split:,:], x_1[split:,:]), dim=0)
y_test = torch.cat((y_0[split:], y_1[split:]), dim=0)

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

In [ ]:
from torch.utils.data import Dataset

class SimpleDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __getitem__(self, index):
        return self.x[index], self.y[index]
    
    def __len__(self):
        return self.x.shape[0]
    

train_ds = SimpleDataset(x_train, y_train)
test_ds = SimpleDataset(x_test, y_test)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
                    dataset=train_ds,
                    batch_size = 4,
                    shuffle=True,
                    drop_last=True
                )
                
test_loader = DataLoader(
                    dataset=test_ds,
                    batch_size=4
                )

In [ ]:
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(d_in, 4*d_in),
            nn.ReLU(),
            nn.Linear(4*d_in, d_out)
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

model = NeuralNetwork(features, 2)
model = model.to(device)
print("Model trainable paramerers:", sum(p.numel() for p in model.parameters() if p.requires_grad))

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

num_epochs = 150
for epoch in range(num_epochs):
    model.train()

    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        
        logits = model(x)
        loss = F.cross_entropy(logits, y)
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d} | Batch: {batch_idx+1:03d}/{len(train_loader):03d} | Loss: {loss:.3f}")


In [ ]:
def compute_accuracy(model, dataloader):
    model.eval()
    with torch.no_grad():
        correct = 0.0
        num_samples = 0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            predictions = torch.argmax(logits, dim=1)
            compare = predictions == y
            correct += torch.sum(compare)
            num_samples += len(compare)

    return (correct / num_samples).item()

In [ ]:
compute_accuracy(model, train_loader)

In [ ]:
compute_accuracy(model, test_loader)